In [1]:
gamma = 0.9
theta = 1e-10

states = ["S0", "S1", "G"]
non_terminal_states = ["S0", "S1"]
actions = ["up", "down", "left", "right"]

# Grid layout:
# S0   S1
# WALL G

grid = [
    ["S0", "S1"],
    ["WALL", "G"]
]

positions = {
    "S0": (0, 0),
    "S1": (0, 1),
    "G": (1, 1)
}

wall = (1, 0)

deltas = {
    "up": (-1, 0),
    "down": (1, 0),
    "left": (0, -1),
    "right": (0, 1)
}

print("STATES:", states)
print("ACTIONS:", actions)
print("Discount factor:", gamma)
print("Rewards: Normal = -1, Wall = -1, Goal = +10")

# Build transition and reward mappings
transition = {}
reward = {}

for s in non_terminal_states:
    r, c = positions[s]

    for a in actions:
        dr, dc = deltas[a]
        nr, nc = r + dr, c + dc

        if (not (0 <= nr < 2 and 0 <= nc < 2)
                or (nr, nc) == wall):
            next_state = s
            immediate_reward = -1
        else:
            next_state = grid[nr][nc]
            immediate_reward = 10 if next_state == "G" else -1

        transition[(s, a)] = next_state
        reward[(s, a)] = immediate_reward

# Verify transitions
print("\nTRANSITION MAPPING")
for s in non_terminal_states:
    for a in actions:
        print(
            f"State: {s}, Action: {a}, "
            f"Next State: {transition[(s, a)]}, "
            f"Reward: {reward[(s, a)]}"
        )

assert len(transition) == len(non_terminal_states) * len(actions)

# Initialize state values
V = {s: 0.0 for s in states}

print("\nINITIAL STATE VALUES")
print(V)

# Synchronous Value Iteration
iteration = 0

print("\nVALUE ITERATION")

while True:
    new_V = V.copy()

    for s in non_terminal_states:
        q_values = []

        for a in actions:
            next_state = transition[(s, a)]
            q = reward[(s, a)] + gamma * V[next_state]
            q_values.append(q)

        new_V[s] = max(q_values)

    new_V["G"] = 0.0

    delta = max(abs(new_V[s] - V[s]) for s in states)
    V = new_V
    iteration += 1

    print(
        f"Iteration {iteration}: "
        f"V(S0)={V['S0']:.6f}, "
        f"V(S1)={V['S1']:.6f}, "
        f"V(G)={V['G']:.6f}, "
        f"Delta={delta:.6f}"
    )

    if delta < theta:
        break

# Final optimal state values
print("\nFINAL OPTIMAL STATE VALUES")
for s in states:
    print(f"V*({s}) = {V[s]:.6f}")

# Calculate action values
Q = {}

print("\nACTION VALUES Q*(s, a)")

for s in non_terminal_states:
    print(f"\nState {s}:")

    for a in actions:
        next_state = transition[(s, a)]
        Q[(s, a)] = (
            reward[(s, a)] + gamma * V[next_state]
        )
        print(f"Q*({s}, {a}) = {Q[(s, a)]:.6f}")

# Extract optimal policy, including ties
print("\nOPTIMAL POLICY")

policy = {}

for s in non_terminal_states:
    max_q = max(Q[(s, a)] for a in actions)

    best_actions = [
        a for a in actions
        if abs(Q[(s, a)] - max_q) < 1e-9
    ]

    policy[s] = best_actions
    print(f"Policy({s}) = {best_actions}")

# Bellman Optimality verification
print("\nBELLMAN OPTIMALITY VERIFICATION")

for s in non_terminal_states:
    max_q = max(Q[(s, a)] for a in actions)
    difference = abs(V[s] - max_q)

    print(
        f"State: {s}, V*={V[s]:.6f}, "
        f"max Q*={max_q:.6f}, "
        f"Difference={difference:.10f}"
    )

    assert difference < 1e-8

print("\nBellman Optimality verification passed.")

STATES: ['S0', 'S1', 'G']
ACTIONS: ['up', 'down', 'left', 'right']
Discount factor: 0.9
Rewards: Normal = -1, Wall = -1, Goal = +10

TRANSITION MAPPING
State: S0, Action: up, Next State: S0, Reward: -1
State: S0, Action: down, Next State: S0, Reward: -1
State: S0, Action: left, Next State: S0, Reward: -1
State: S0, Action: right, Next State: S1, Reward: -1
State: S1, Action: up, Next State: S1, Reward: -1
State: S1, Action: down, Next State: G, Reward: 10
State: S1, Action: left, Next State: S0, Reward: -1
State: S1, Action: right, Next State: S1, Reward: -1

INITIAL STATE VALUES
{'S0': 0.0, 'S1': 0.0, 'G': 0.0}

VALUE ITERATION
Iteration 1: V(S0)=-1.000000, V(S1)=10.000000, V(G)=0.000000, Delta=10.000000
Iteration 2: V(S0)=8.000000, V(S1)=10.000000, V(G)=0.000000, Delta=9.000000
Iteration 3: V(S0)=8.000000, V(S1)=10.000000, V(G)=0.000000, Delta=0.000000

FINAL OPTIMAL STATE VALUES
V*(S0) = 8.000000
V*(S1) = 10.000000
V*(G) = 0.000000

ACTION VALUES Q*(s, a)

State S0:
Q*(S0, up) = 6.2